# TitlePulse Phase 2: Exploratory Data Analysis

This notebook loads the raw video data from Phase 1, performs basic cleaning, and explores the distributions of key variables like views, subscribers, and channels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# Set plot style
sns.set_theme(style="whitegrid")

## 1. Load Data

Load all raw CSV files into a single DataFrame.

In [ ]:
raw_dir = '../data/raw'
csv_files = glob.glob(os.path.join(raw_dir, '*.csv'))

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    dfs.append(df)
    
df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(df)} total videos from {len(csv_files)} files.")

## 2. Basic Dataset Summary

Let's look at total videos, total unique channels, and breakdowns by `niche_tags` and `size_tier`.

In [ ]:
print(f"Total Videos: {len(df)}")
print(f"Total Unique Channels: {df['channel_id'].nunique()}")
print("\n--- Breakdown by Size Tier ---")
print(df['size_tier'].value_counts())
print("\n--- Breakdown by Niche Tags ---")
print(df['niche_tags'].value_counts())

## 3. Data Quality & Malformed Rows

Identify broken rows: missing titles, 0 views, duplicate video_ids, missing published_at.

In [ ]:
missing_titles = df['title'].isna().sum()
zero_views = (df['view_count'] <= 0).sum() | df['view_count'].isna().sum()
missing_pub = df['published_at'].isna().sum()
duplicates = df.duplicated(subset=['video_id']).sum()

print(f"Missing Titles: {missing_titles}")
print(f"Zero or Missing Views: {zero_views}")
print(f"Missing Published Date: {missing_pub}")
print(f"Duplicate Video IDs: {duplicates}")

Let's drop these for the rest of the visual analysis.

In [ ]:
df_clean = df.dropna(subset=['title', 'published_at']).copy()
df_clean = df_clean[df_clean['view_count'] > 0]
df_clean = df_clean.drop_duplicates(subset=['video_id'])
print(f"Videos remaining after cleaning: {len(df_clean)}")

## 4. Distribution of Raw View Count

This distribution is expected to be heavily right-skewed, as a few videos get massive amounts of views while most get very few.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_clean['view_count'], bins=50, kde=True)
plt.title('Distribution of Raw View Count')
plt.xlabel('View Count')
plt.ylabel('Frequency')
plt.show()

## 5. Distribution of Log1p View Count

Applying a `log1p` transform (log(1 + x)) should make the distribution look much closer to normal, which is better for modeling.

In [ ]:
df_clean['log_view_count'] = np.log1p(df_clean['view_count'])
plt.figure(figsize=(10, 6))
sns.histplot(df_clean['log_view_count'], bins=50, kde=True)
plt.title('Distribution of Log1p View Count')
plt.xlabel('Log1p(View Count)')
plt.ylabel('Frequency')
plt.show()

## 6. Distribution of Subscriber Count Across Channels

Let's see the distribution of channel sizes.

In [ ]:
channel_df = df_clean.drop_duplicates(subset=['channel_id'])
plt.figure(figsize=(10, 6))
sns.histplot(channel_df['subscriber_count'], bins=50, kde=True)
plt.title('Distribution of Subscriber Count (Unique Channels)')
plt.xlabel('Subscriber Count')
plt.ylabel('Frequency')
plt.show()

## 7. Subscribers vs. Views (Log-Log Scale)

Do bigger channels get more views? Let's check the correlation. We use a log-log scale due to the wide variance.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x='subscriber_count', y='view_count', alpha=0.3)
plt.xscale('log')
plt.yscale('log')
plt.title('Subscriber Count vs. View Count (Log-Log Scale)')
plt.xlabel('Subscriber Count (Log)')
plt.ylabel('View Count (Log)')
plt.show()

## 8. Per-Channel Video Count Distribution

We need to flag channels with too few videos (e.g., < 5), as their "rolling average" baseline won't be reliable.

In [ ]:
video_counts = df_clean.groupby('channel_id')['video_id'].count()

plt.figure(figsize=(10, 6))
sns.histplot(video_counts, bins=range(1, video_counts.max() + 2), discrete=True)
plt.title('Distribution of Videos per Channel')
plt.xlabel('Number of Videos')
plt.ylabel('Frequency (Channels)')
plt.xlim(0, 50)
plt.show()

low_video_channels = (video_counts < 5).sum()
print(f"Channels with fewer than 5 videos: {low_video_channels} out of {len(video_counts)} ({(low_video_channels/len(video_counts))*100:.2f}%)")